In [1]:
import json
import re

def convert_notebook(filepath, filename):
    with open(filepath, 'r') as f:
        notebook = json.load(f)
    ## keep this top section

    html = '<html><body>'
    skip_until_next_heading = False
    current_heading_level = None
    
    for cell in notebook['cells']:
        # Skip hidden cells
        if cell.get('metadata', {}).get('jupyter', {}).get('source_hidden', False):
            continue
        
        source = ''.join(cell.get('source', []))
        
        # Check if this is a markdown heading and extract level
        heading_match = re.match(r'^(#+)\s', source)
        is_heading = heading_match is not None
        
        if is_heading:
            heading_level = len(heading_match.group(1))
            
            # If we're skipping and hit a heading at same or higher level, stop skipping
            if skip_until_next_heading and current_heading_level is not None:
                if heading_level <= current_heading_level:
                    skip_until_next_heading = False
            
            # Check if this heading is collapsed
            if cell.get('metadata', {}).get('jp-MarkdownHeadingCollapsed', False):
                skip_until_next_heading = True
                current_heading_level = heading_level
                continue
        
        # Skip cells under a collapsed heading
        if skip_until_next_heading:
            continue
        
        if cell['cell_type'] == 'markdown':
            html += '<div>' + source + '</div>'
        elif cell['cell_type'] == 'code':
            html += '<pre><code>' + source + '</code></pre>'
            
            # Process outputs
            for output in cell.get('outputs', []):
                if output.get('output_type') == 'stream':
                    text = ''.join(output.get('text', []))
                    html += '<pre>' + text + '</pre>'
                elif output.get('output_type') == 'display_data':
                    data = output.get('data', {})
                    if 'image/png' in data:
                        img_data = data['image/png']
                        html += f'<img src="data:image/png;base64,{img_data}" />'
                    elif 'image/jpeg' in data:
                        img_data = data['image/jpeg']
                        html += f'<img src="data:image/jpeg;base64,{img_data}" />'
    
    html += '</body></html>'
    
    with open(f'{filename}.html', 'w') as f:
        f.write(html)

In [4]:
filepath = 'forecast-analysis/forecast_analysis.ipynb'
filename = 'item sales charts'
convert_notebook(filepath, filename)